In [2]:
import pandas as pd
import numpy as np

def price_storage_contract(
    injection_dates,      # List/Array of dates gas is purchased and injected
    withdrawal_dates,     # List/Array of dates gas is withdrawn and sold
    injection_rate,       # Volume of gas injected per day (e.g., MMBtu/day)
    withdrawal_rate,      # Volume of gas withdrawn per day (e.g., MMBtu/day)
    max_capacity,         # Maximum storage capacity allowed (e.g., MMBtu)
    storage_cost_per_day, # Fixed facility storage cost per day ($)
    injection_cost_rate,  # Variable cost per unit volume of gas injected ($/MMBtu)
    withdrawal_cost_rate, # Variable cost per unit volume of gas withdrawn ($/MMBtu)
    price_getter_func     # Function or lookup mapping (Date -> Price)
):
    """
    Calculates the Net Present Value (Contract Value) of a natural gas storage contract.
    Assumes 0% interest rate and no transport delays.
    """

    # 1. Calculate Injections (Cash Outflow)
    total_injected_volume = 0
    total_injection_cost = 0

    for date in injection_dates:
        # Check storage capacity limit
        if total_injected_volume + injection_rate > max_capacity:
            raise ValueError(f"Storage capacity of {max_capacity} exceeded on {date}")

        gas_price = price_getter_func(date)
        purchase_amount = injection_rate * gas_price
        pumping_fee = injection_rate * injection_cost_rate

        total_injection_cost += (purchase_amount + pumping_fee)
        total_injected_volume += injection_rate

    # 2. Calculate Withdrawals (Cash Inflow)
    total_withdrawn_volume = 0
    total_withdrawal_revenue = 0

    for date in withdrawal_dates:
        # Ensure we don't withdraw more gas than is stored
        if total_withdrawn_volume + withdrawal_rate > total_injected_volume:
            raise ValueError(f"Cannot withdraw {withdrawal_rate} on {date}. Insufficient inventory.")

        gas_price = price_getter_func(date)
        sales_revenue = withdrawal_rate * gas_price
        pumping_fee = withdrawal_rate * withdrawal_cost_rate

        total_withdrawal_revenue += (sales_revenue - pumping_fee)
        total_withdrawn_volume += withdrawal_rate

    # 3. Calculate Total Days in Storage Facility (Rental Duration)
    all_dates = pd.to_datetime(injection_dates + withdrawal_dates)
    start_date = all_dates.min()
    end_date = all_dates.max()
    total_days = (end_date - start_date).days + 1

    total_fixed_storage_cost = total_days * storage_cost_per_day

    # 4. Calculate Final Contract Net Value
    net_contract_value = total_withdrawal_revenue - total_injection_cost - total_fixed_storage_cost

    # Print Detailed Cash Flow Summary
    print("===== CONTRACT VALUATION SUMMARY =====")
    print(f"Total Volume Injected:  {total_injected_volume:,.2f} MMBtu")
    print(f"Total Volume Withdrawn: {total_withdrawn_volume:,.2f} MMBtu")
    print(f"Storage Duration:       {total_days} days")
    print("--------------------------------------")
    print(f"Gross Gas Purchase:    -${(total_injection_cost):,.2f}")
    print(f"Gross Gas Revenue:     +${(total_withdrawal_revenue):,.2f}")
    print(f"Fixed Storage Rent:    -${total_fixed_storage_cost:,.2f}")
    print("--------------------------------------")
    print(f"NET CONTRACT VALUE:     ${net_contract_value:,.2f}")
    print("======================================")

    return net_contract_value

    # Dummy Price Getter function simulating low summer prices & high winter prices
def mock_price_lookup(date_str):
    prices = {
        '2024-06-01': 2.00,  # Summer Buy
        '2024-06-02': 2.10,  # Summer Buy
        '2024-12-01': 3.50,  # Winter Sell
        '2024-12-02': 3.60   # Winter Sell
    }
    return prices.get(date_str, 2.50)  # Default fallback price

# --- TEST SAMPLE INPUTS ---
inj_dates = ['2024-06-01', '2024-06-02']
with_dates = ['2024-12-01', '2024-12-02']

inj_rate = 500000      # 500,000 MMBtu per injection date (Total 1M MMBtu)
with_rate = 500000     # 500,000 MMBtu per withdrawal date
max_cap = 1500000      # Storage capacity limit
storage_rent = 1000    # $1,000 fixed rent per day
inj_fee = 0.01         # $0.01 per MMBtu variable injection fee
with_fee = 0.01        # $0.01 per MMBtu variable withdrawal fee

# Run Valuation
contract_value = price_storage_contract(
    injection_dates=inj_dates,
    withdrawal_dates=with_dates,
    injection_rate=inj_rate,
    withdrawal_rate=with_rate,
    max_capacity=max_cap,
    storage_cost_per_day=storage_rent,
    injection_cost_rate=inj_fee,
    withdrawal_cost_rate=with_fee,
    price_getter_func=mock_price_lookup
)

===== CONTRACT VALUATION SUMMARY =====
Total Volume Injected:  1,000,000.00 MMBtu
Total Volume Withdrawn: 1,000,000.00 MMBtu
Storage Duration:       185 days
--------------------------------------
Gross Gas Purchase:    -$2,060,000.00
Gross Gas Revenue:     +$3,540,000.00
Fixed Storage Rent:    -$185,000.00
--------------------------------------
NET CONTRACT VALUE:     $1,295,000.00
